# BD-KDD Kidney Disease — ANOVA + DODA Feature Selection

## Experimental objective

Evaluate conventional **ANOVA feature selection** against **ANOVA + DODA clinical-priority re-ranking** on the BD-KDD kidney-disease dataset.

### Important design rule

ANOVA scores **all 24 predictors**. DODA also receives **all 24 predictor scores** and performs the final Top-K selection after clinical-priority fusion.

This is essential: if ANOVA selected Top-K before DODA, DODA could not re-prioritize features outside that preliminary Top-K.

### Dataset

- 988 records
- 24 predictors
- Binary target: `Class`
- `Class = 0`: healthy
- `Class = 1`: kidney disease
- 0 missing values
- 80/20 stratified fixed split for the initial experiment
- 5×5 repeated stratified CV for robustness


In [1]:
# =============================================================================
# STEP 1: LOAD PROCESSED DATASET
# =============================================================================

import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path("../../data/processed/BD-KDD_cleaned.csv")

if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH.resolve()}"
    )

df = pd.read_csv(DATA_PATH)

print("=" * 70)
print("PROCESSED BD-KDD DATASET")
print("=" * 70)
print(f"Rows    : {df.shape[0]}")
print(f"Columns : {df.shape[1]}")

display(df.head())


PROCESSED BD-KDD DATASET
Rows    : 988
Columns : 25


,Age,Bp,Sg,Al,Su,Rbc,Pc,Pcc,Ba,Bgr,...,Pcv,Wbcc,Rbcc,Htn,Dm,Cad,Appet,Pe,Ane,Class
0,58,147,1.025,1,3,0,1,0,0,177,...,43,12342,4.3,1,0,0,1,1,0,0
1,71,142,1.025,2,3,1,0,1,0,247,...,47,7249,3.7,0,1,1,0,1,1,1
2,48,179,1.005,0,4,1,1,0,1,367,...,37,4111,3.7,0,0,0,0,0,1,0
3,34,70,1.005,1,3,1,1,0,0,215,...,44,4682,4.0,1,0,1,1,0,1,1
4,62,120,1.020,4,0,1,0,1,0,143,...,54,5161,4.7,0,0,1,0,1,0,1


In [2]:
# =============================================================================
# STEP 2: FEATURE AND TARGET SEPARATION
# =============================================================================

TARGET_COLUMN = "Class"

X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

print("=" * 70)
print("FEATURE / TARGET SEPARATION")
print("=" * 70)
print(f"Predictors (X) : {X.shape[1]}")
print(f"Records        : {X.shape[0]}")
print(f"Target (y)     : {TARGET_COLUMN}")
print(f"Target values  : {sorted(y.unique().tolist())}")

assert X.shape[1] == 24, "Expected 24 predictors after removing Class."
assert set(y.unique()) == {0, 1}

display(X.head())
display(y.head())


FEATURE / TARGET SEPARATION
Predictors (X) : 24
Records        : 988
Target (y)     : Class
Target values  : [0, 1]


,Age,Bp,Sg,Al,Su,Rbc,Pc,Pcc,Ba,Bgr,...,Hemo,Pcv,Wbcc,Rbcc,Htn,Dm,Cad,Appet,Pe,Ane
0,58,147,1.025,1,3,0,1,0,0,177,...,9.3,43,12342,4.3,1,0,0,1,1,0
1,71,142,1.025,2,3,1,0,1,0,247,...,16.9,47,7249,3.7,0,1,1,0,1,1
2,48,179,1.005,0,4,1,1,0,1,367,...,16.6,37,4111,3.7,0,0,0,0,0,1
3,34,70,1.005,1,3,1,1,0,0,215,...,15.4,44,4682,4.0,1,0,1,1,0,1
4,62,120,1.020,4,0,1,0,1,0,143,...,13.9,54,5161,4.7,0,0,1,0,1,0


0    0
1    1
2    0
3    1
4    1
Name: Class, dtype: int64

In [3]:
# =============================================================================
# STEP 3: TARGET DISTRIBUTION
# =============================================================================

target_distribution = pd.DataFrame({
    "Count": y.value_counts().sort_index(),
    "Percentage": (y.value_counts(normalize=True).sort_index() * 100).round(2)
})

display(target_distribution)


,Count,Percentage
Class,,
0,481,48.68
1,507,51.32


In [4]:
# =============================================================================
# STEP 4: DATA TYPE AND MISSING-VALUE CHECK
# =============================================================================

print("=" * 70)
print("DATA TYPES")
print("=" * 70)
display(X.dtypes.to_frame("Data Type"))

missing = pd.DataFrame({
    "Missing Count": X.isna().sum(),
    "Missing Percentage (%)": (X.isna().mean() * 100).round(2)
})

print("=" * 70)
print("MISSING VALUES")
print("=" * 70)
display(missing)

assert X.isna().sum().sum() == 0, "Unexpected missing values detected."

print("No missing values detected. No imputation will be performed.")


DATA TYPES


,Data Type
Age,int64
Bp,int64
Sg,float64
Al,int64
Su,int64
Rbc,int64
Pc,int64
Pcc,int64
Ba,int64
Bgr,int64


MISSING VALUES


,Missing Count,Missing Percentage (%)
Age,0,0.0
Bp,0,0.0
Sg,0,0.0
Al,0,0.0
Su,0,0.0
Rbc,0,0.0
Pc,0,0.0
Pcc,0,0.0
Ba,0,0.0
Bgr,0,0.0


No missing values detected. No imputation will be performed.


## Preprocessing decision

The EDA established that BD-KDD contains **zero missing values**. Therefore, no imputation step is included in this notebook.

Feature selection is fitted only on training data/folds. Model scaling is applied only where needed and is fitted only on training data/folds.


In [5]:
# =============================================================================
# STEP 5: FIXED 80/20 STRATIFIED TRAIN-TEST SPLIT
# =============================================================================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("=" * 70)
print("TRAIN-TEST SPLIT")
print("=" * 70)
print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"y_train : {y_train.shape}")
print(f"y_test  : {y_test.shape}")

print("\nTraining class distribution:")
display((y_train.value_counts(normalize=True) * 100).round(2))

print("\nTesting class distribution:")
display((y_test.value_counts(normalize=True) * 100).round(2))


TRAIN-TEST SPLIT
X_train : (790, 24)
X_test  : (198, 24)
y_train : (790,)
y_test  : (198,)

Training class distribution:


Class
1    51.27
0    48.73
Name: proportion, dtype: float64


Testing class distribution:


Class
1    51.52
0    48.48
Name: proportion, dtype: float64

In [6]:
# =============================================================================
# STEP 6: FEATURE-SCALE CHECK
# =============================================================================
# ANOVA F-statistics are scale-invariant, so feature selection is performed
# on the original training-fold values.
#
# Scaling is applied later only for Logistic Regression.

print("ANOVA will be fitted on the original training data.")
print("Logistic Regression will be scaled after feature selection.")
print("Random Forest and XGBoost will use the selected features without scaling.")


ANOVA will be fitted on the original training data.
Logistic Regression will be scaled after feature selection.
Random Forest and XGBoost will use the selected features without scaling.


# 1. ANOVA Baseline

ANOVA is used as the statistical feature-selection baseline.

**Important:** ANOVA scores all 24 predictors first. Top-K is applied only after the complete ranking has been generated.


In [7]:
# =============================================================================
# STEP 7: ANOVA SCORES FOR ALL FEATURES
# =============================================================================

from sklearn.feature_selection import SelectKBest, f_classif

anova_selector = SelectKBest(
    score_func=f_classif,
    k="all"
)

anova_selector.fit(X_train, y_train)

anova_scores = pd.DataFrame({
    "Feature": X_train.columns,
    "ANOVA_F_Score": anova_selector.scores_,
    "ANOVA_P_Value": anova_selector.pvalues_
}).sort_values(
    "ANOVA_F_Score",
    ascending=False
).reset_index(drop=True)

anova_scores["ANOVA_Rank"] = np.arange(1, len(anova_scores) + 1)

display(anova_scores)


,Feature,ANOVA_F_Score,ANOVA_P_Value,ANOVA_Rank
0,Bp,8.902625,0.002936,1
1,Su,1.475878,0.224784,2
2,Ba,1.387509,0.239182,3
3,Bu,1.295059,0.255464,4
4,Ane,1.236786,0.266431,5
5,Rbcc,0.835335,0.361014,6
6,Pcv,0.779051,0.377701,7
7,Sc,0.770546,0.380316,8
8,Appet,0.634996,0.425769,9
9,Bgr,0.596137,0.440287,10


In [8]:
# =============================================================================
# STEP 8: TOP-K ANOVA FEATURE SETS
# =============================================================================

TOP_K_VALUES = [5, 10, 15, 20]

print(f"Total predictors: {X.shape[1]}")
print(f"Top-K values: {TOP_K_VALUES}")

anova_results = {}

for top_k in TOP_K_VALUES:

    selected = anova_scores.head(top_k)["Feature"].tolist()

    anova_results[top_k] = {
        "features": selected,
        "X_train": X_train[selected].copy(),
        "X_test": X_test[selected].copy()
    }

    print("=" * 70)
    print(f"ANOVA TOP-{top_k}")
    print("=" * 70)
    print(selected)


Total predictors: 24
Top-K values: [5, 10, 15, 20]
ANOVA TOP-5
['Bp', 'Su', 'Ba', 'Bu', 'Ane']
ANOVA TOP-10
['Bp', 'Su', 'Ba', 'Bu', 'Ane', 'Rbcc', 'Pcv', 'Sc', 'Appet', 'Bgr']
ANOVA TOP-15
['Bp', 'Su', 'Ba', 'Bu', 'Ane', 'Rbcc', 'Pcv', 'Sc', 'Appet', 'Bgr', 'Sg', 'Dm', 'Pcc', 'Pe', 'Al']
ANOVA TOP-20
['Bp', 'Su', 'Ba', 'Bu', 'Ane', 'Rbcc', 'Pcv', 'Sc', 'Appet', 'Bgr', 'Sg', 'Dm', 'Pcc', 'Pe', 'Al', 'Pot', 'Wbcc', 'Cad', 'Rbc', 'Htn']


In [9]:
# =============================================================================
# STEP 9: DOWNSTREAM MODELS
# =============================================================================

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )
}

print(list(models.keys()))


['Logistic Regression', 'Random Forest', 'XGBoost']


In [10]:
# =============================================================================
# STEP 10: ANOVA BASELINE MODEL EVALUATION
# =============================================================================

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

fixed_anova_rows = []

for top_k in TOP_K_VALUES:

    Xtr_selected = anova_results[top_k]["X_train"]
    Xte_selected = anova_results[top_k]["X_test"]
    selected_features = anova_results[top_k]["features"]

    for model_name, model_template in models.items():

        model = model_template.__class__(
            **model_template.get_params()
        )

        Xtr = Xtr_selected
        Xte = Xte_selected

        if model_name == "Logistic Regression":
            scaler = StandardScaler()
            Xtr = scaler.fit_transform(Xtr)
            Xte = scaler.transform(Xte)

        model.fit(Xtr, y_train)

        y_pred = model.predict(Xte)
        y_prob = model.predict_proba(Xte)[:, 1]

        fixed_anova_rows.append({
            "Method": "ANOVA",
            "Top_K": top_k,
            "Model": model_name,
            "Selected_Features": ", ".join(selected_features),
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred, zero_division=0),
            "Recall": recall_score(y_test, y_pred, zero_division=0),
            "F1": f1_score(y_test, y_pred, zero_division=0),
            "ROC_AUC": roc_auc_score(y_test, y_prob)
        })

fixed_anova_results_df = pd.DataFrame(fixed_anova_rows)
display(fixed_anova_results_df.round(4))


,Method,Top_K,Model,Selected_Features,Accuracy,Precision,Recall,F1,ROC_AUC
0,ANOVA,5,Logistic Regression,"Bp, Su, Ba, Bu, Ane",0.5303,0.5391,0.6078,0.5714,0.5587
1,ANOVA,5,Random Forest,"Bp, Su, Ba, Bu, Ane",0.5303,0.5474,0.5098,0.5279,0.5600
2,ANOVA,5,XGBoost,"Bp, Su, Ba, Bu, Ane",0.5000,0.5143,0.5294,0.5217,0.5036
3,ANOVA,10,Logistic Regression,"Bp, Su, Ba, Bu, Ane, Rbcc, Pcv, Sc, Appet, Bgr",0.5606,0.5630,0.6569,0.6063,0.5895
4,ANOVA,10,Random Forest,"Bp, Su, Ba, Bu, Ane, Rbcc, Pcv, Sc, Appet, Bgr",0.5051,0.5208,0.4902,0.5051,0.5129
5,ANOVA,10,XGBoost,"Bp, Su, Ba, Bu, Ane, Rbcc, Pcv, Sc, Appet, Bgr",0.4848,0.5000,0.4804,0.4900,0.4931
6,ANOVA,15,Logistic Regression,"Bp, Su, Ba, Bu, Ane, Rbcc, Pcv, Sc, Appet, Bgr...",0.5556,0.5660,0.5882,0.5769,0.5684
7,ANOVA,15,Random Forest,"Bp, Su, Ba, Bu, Ane, Rbcc, Pcv, Sc, Appet, Bgr...",0.5556,0.5686,0.5686,0.5686,0.5510
8,ANOVA,15,XGBoost,"Bp, Su, Ba, Bu, Ane, Rbcc, Pcv, Sc, Appet, Bgr...",0.5051,0.5204,0.5000,0.5100,0.5314
9,ANOVA,20,Logistic Regression,"Bp, Su, Ba, Bu, Ane, Rbcc, Pcv, Sc, Appet, Bgr...",0.5505,0.5596,0.5980,0.5782,0.5751


In [11]:
# =============================================================================
# STEP 11: SAVE ANOVA BASELINE
# =============================================================================

baseline_dir = Path("../../../results/kidney_disease/baseline")
baseline_dir.mkdir(parents=True, exist_ok=True)

baseline_path = baseline_dir / "anova_baseline_results.csv"
fixed_anova_results_df.to_csv(baseline_path, index=False)

print(f"Saved: {baseline_path.resolve()}")


Saved: /home/claude/bd_kdd_final/results/kidney_disease/baseline/anova_baseline_results.csv


# 2. DODA Clinical Re-ranking

### Correct experimental order

```text
24 original predictors
        ↓
ANOVA scores all 24
        ↓
statistical ranking
        +
pre-specified clinical weights
        ↓
DODA rank fusion
        ↓
final Top-K
```

DODA must **not** receive an already-truncated ANOVA Top-K set. Otherwise, clinically prioritized features outside that preliminary Top-K would be impossible to recover.


In [12]:
# =============================================================================
# STEP 12: INSTALL / IMPORT DODA
# =============================================================================

# Run the following installation once if DODA is not installed:
# %pip install --no-cache-dir git+https://github.com/anandha-3679/DODA.git

import doda

from doda import DODASelector
from doda.adapters import SklearnAdapter
from doda.knowledge.providers import JSONProvider
from doda.fusion import RankFusion

print("DODA:", doda.__file__)

print("Provider:", JSONProvider)
print("Fusion:", RankFusion)

DODA: /usr/local/lib/python3.12/dist-packages/doda/__init__.py
Provider: <class 'doda.knowledge.providers.json_provider.JSONProvider'>
Fusion: <class 'doda.fusion.rank.RankFusion'>


### Why a custom provider is needed here

`BD_KDD_clinical_weights.json` uses a richer schema than the package's `JSONProvider` expects — the per-feature weights live under a nested `"weights"` key, alongside `schema_version`, `sources`, `feature_tiers`, and other provenance metadata. `JSONProvider.get_weights()` only ever looks at the **top level** of the loaded JSON, so for every feature it does `self.weights.get("Age", 1.0)` — and since `"Age"` is not a top-level key in this file, **every single feature silently fell back to the default weight of 1.0**. This made the clinical layer completely flat (a tie across all 24 features), which is exactly why DODA wasn't producing any rank changes: Rank Fusion had a real statistical ranking to combine with, but a clinical ranking with no information in it at all.

This was previously masked by the validation cell below, which manually did `weights_payload["weights"]` to check coverage — that check is correct, but it validates a *different* object than the one actually passed into `DODASelector`, so it passed even though the real `provider` was broken.

**Fix**: `NestedJSONProvider`, defined below, implements the same `BaseKnowledgeProvider` interface as `JSONProvider` but reads from the nested `"weights"` key. The clinical weights file itself does not need to change — it's correctly structured for the richer schema; the loading code needed to catch up to it. If this schema becomes the standard going forward, `JSONProvider` in the actual `doda` package should be updated to check for a `"weights"` key and fall back to top-level parsing for older flat files, so every notebook doesn't need its own local copy of this class.

In [13]:
# =============================================================================
# STEP 12B: NESTED-SCHEMA CLINICAL KNOWLEDGE PROVIDER
# =============================================================================

from doda.knowledge.providers.base import BaseKnowledgeProvider
import json as _json


class NestedJSONProvider(BaseKnowledgeProvider):
    """
    Reads clinical weights from the "weights" key of a richer JSON schema
    (schema_version, target, tier_mapping, feature_tiers, sources, etc.),
    rather than expecting the weights at the top level the way the
    package's built-in JSONProvider does.

    Same fallback behavior as JSONProvider otherwise: a feature not present
    in the weights dict gets a neutral default of 1.0, not an error --
    kept intentionally consistent so debugging behavior doesn't change,
    only where the weights are actually read from.
    """

    def __init__(self, path):
        self.path = path
        with open(path, "r", encoding="utf-8") as f:
            payload = _json.load(f)

        if "weights" not in payload:
            raise KeyError(
                f"{path} has no top-level 'weights' key -- "
                "this loader expects the nested schema. Use JSONProvider "
                "instead for a flat {feature: weight} file."
            )

        self.weights = payload["weights"]

    def get_weights(self, feature_names):
        return {
            feature: self.weights.get(feature, 1.0)
            for feature in feature_names
        }


print("NestedJSONProvider defined.")

NestedJSONProvider defined.


In [14]:
# =============================================================================
# STEP 13: LOCATE AND VALIDATE CLINICAL WEIGHTS
# =============================================================================

WEIGHTS_FILENAME = "BD_KDD_clinical_weights.json"

search_roots = [Path.cwd(), *Path.cwd().parents]
matches = []

for root in search_roots:
    try:
        matches.extend(root.rglob(WEIGHTS_FILENAME))
    except (PermissionError, OSError):
        continue

# Remove duplicates while preserving order
matches = list(dict.fromkeys(matches))

if not matches:
    raise FileNotFoundError(
        f"{WEIGHTS_FILENAME} was not found. "
        "Place it in the repository and rerun this cell."
    )

WEIGHTS_PATH = matches[0]

print(f"Using clinical weights: {WEIGHTS_PATH.resolve()}")

provider = NestedJSONProvider(str(WEIGHTS_PATH))

# Validate coverage using the JSON file directly.
import json

with open(WEIGHTS_PATH, "r", encoding="utf-8") as f:
    weights_payload = json.load(f)

clinical_weights = weights_payload["weights"]

missing_weights = sorted(set(X.columns) - set(clinical_weights))
extra_weights = sorted(set(clinical_weights) - set(X.columns))

print(f"Clinical weights: {len(clinical_weights)}")
print(f"Predictors:       {len(X.columns)}")
print(f"Missing weights:  {missing_weights}")
print(f"Extra weights:    {extra_weights}")

assert not missing_weights
assert not extra_weights


Using clinical weights: /home/claude/bd_kdd_final/notebooks/angel/kidney_disease/BD_KDD_clinical_weights.json
Clinical weights: 24
Predictors:       24
Missing weights:  []
Extra weights:    []


In [15]:
# =============================================================================
# STEP 14: DODA — SCORE ALL FEATURES, THEN APPLY TOP-K
# =============================================================================

rank_doda_results = {}

for top_k in TOP_K_VALUES:

    anova_operator = SklearnAdapter(
        SelectKBest(
            score_func=f_classif,
            k="all"     # CRITICAL: all 24 features reach DODA
        )
    )

    doda_selector = DODASelector(
        operators=[anova_operator],
        provider=provider,
        fusion=RankFusion(),
        top_k=top_k
    )

    # Fit only on training data.
    doda_selector.fit(
        X_train,
        y_train
    )

    selected_features = list(
        doda_selector.get_selected_features()
    )

    rank_doda_results[top_k] = {
        "selector": doda_selector,
        "features": selected_features,
        "X_train": X_train[selected_features].copy(),
        "X_test": X_test[selected_features].copy()
    }

    print("=" * 70)
    print(f"DODA TOP-{top_k}")
    print("=" * 70)
    print(selected_features)
    print(f"Number selected: {len(selected_features)}")

    assert len(selected_features) == top_k


Provider inside knowledge engine: <__main__.NestedJSONProvider object at 0x7f588a5b0ef0>
DODA Pipeline Started
Running: SelectKBest
DODA feature order:
['Age', 'Bp', 'Sg', 'Al', 'Su', 'Rbc', 'Pc', 'Pcc', 'Ba', 'Bgr']
Number of features: 24
Number of scores: 24
Operator Scores:
{'SelectKBest_1': {'Age': np.float64(0.015848009107691952), 'Bp': np.float64(8.902625130307502), 'Sg': np.float64(0.5823525344971223), 'Al': np.float64(0.16292862378792028), 'Su': np.float64(1.4758784262753983), 'Rbc': np.float64(0.08822043465353854), 'Pc': np.float64(8.098897814608765e-05), 'Pcc': np.float64(0.4320562634767095), 'Ba': np.float64(1.387509378150789), 'Bgr': np.float64(0.5961366416517426), 'Bu': np.float64(1.2950594931817596), 'Sc': np.float64(0.7705460731720983), 'Sod': np.float64(0.0005941532857815975), 'Pot': np.float64(0.14517187640480858), 'Hemo': np.float64(0.05114806428390342), 'Pcv': np.float64(0.7790510482306884), 'Wbcc': np.float64(0.10643069394407427), 'Rbcc': np.float64(0.83533494915942

In [16]:
# =============================================================================
# STEP 15: INSPECT DODA RANK CHANGES
# =============================================================================

INSPECTION_TOP_K = 10

selector = rank_doda_results[INSPECTION_TOP_K]["selector"]

print("=" * 70)
print(f"DODA RANK COMPARISON — TOP-{INSPECTION_TOP_K}")
print("=" * 70)

rank_comparison = selector.compare_scores()

display(rank_comparison)

print("\nSelected features:")
print(rank_doda_results[INSPECTION_TOP_K]["features"])


DODA RANK COMPARISON — TOP-10

MATHEMATICAL vs CLINICAL vs RANKFUSION

Fusion Method: RankFusion

Top Features After Fusion:
  Feature  Math Rank  Clinical Rank  Final Rank  Rank Change
0      Bp          1              6           1            0
1      Bu          4              3           2            2
2      Sc          8              2           3            5
3      Su          2             16           4           -2
4     Ane          5             11           5            0
5     Pcv          7             10           6            1
6      Sg         11              4           7            4
7      Ba          3             22           8           -5
8    Rbcc          6             18           9           -3
9      Al         15              1          10            5


,Feature,Math Rank,Normalized Math Score,Clinical Rank,Clinical Weight,Final Rank,Final Score,Rank Change
0,Bp,1,1.0000,6,0.80,1,0.0320,0
1,Bu,4,0.1455,3,0.90,2,0.0318,2
2,Sc,8,0.0866,2,1.00,3,0.0311,5
3,Su,2,0.1658,16,0.55,4,0.0308,-2
4,Ane,5,0.1389,11,0.70,5,0.0305,0
5,Pcv,7,0.0875,10,0.70,6,0.0301,1
6,Sg,11,0.0654,4,0.85,7,0.0300,4
7,Ba,3,0.1559,22,0.40,8,0.0300,-5
8,Rbcc,6,0.0938,18,0.55,9,0.0299,-3
9,Al,15,0.0183,1,1.00,10,0.0297,5



Selected features:
['Bp', 'Bu', 'Sc', 'Su', 'Ane', 'Pcv', 'Sg', 'Ba', 'Rbcc', 'Al']


In [17]:
# =============================================================================
# STEP 15B: EXPLICIT ANOVA vs DODA RANK-SHIFT TABLE
# =============================================================================
# This table separates:
#   - statistical ANOVA rank
#   - DODA final rank
#   - clinical weight
#   - rank movement
#
# Positive Rank_Change means the feature moved upward.
# Example: ANOVA rank 10 -> DODA rank 4 = +6
#
# IMPORTANT:
# The selector is explicitly retrieved from rank_doda_results
# for the selected Top-K value. This avoids relying on a
# previously defined 'selector' variable.

# -------------------------------------------------------------------------
# Select which DODA Top-K experiment to inspect
# -------------------------------------------------------------------------

k = 10

if k not in rank_doda_results:
    raise KeyError(
        f"Top-K={k} was not found in rank_doda_results. "
        f"Available values: {list(rank_doda_results.keys())}"
    )

selector = rank_doda_results[k]["selector"]


# -------------------------------------------------------------------------
# ANOVA / mathematical ranking
# -------------------------------------------------------------------------

math_rank = pd.DataFrame(
    selector.math_scores_.items(),
    columns=[
        "Feature",
        "ANOVA_Normalized_Score"
    ]
)

math_rank["ANOVA_Rank"] = (
    math_rank["ANOVA_Normalized_Score"]
    .rank(
        method="min",
        ascending=False
    )
    .astype(int)
)


# -------------------------------------------------------------------------
# DODA final ranking
# -------------------------------------------------------------------------

final_rank = pd.DataFrame(
    selector.final_scores_.items(),
    columns=[
        "Feature",
        "DODA_Final_Score"
    ]
)

final_rank["DODA_Rank"] = (
    final_rank["DODA_Final_Score"]
    .rank(
        method="min",
        ascending=False
    )
    .astype(int)
)


# -------------------------------------------------------------------------
# Combine rankings
# -------------------------------------------------------------------------

rank_shift_df = (
    math_rank[
        [
            "Feature",
            "ANOVA_Normalized_Score",
            "ANOVA_Rank"
        ]
    ]
    .merge(
        final_rank[
            [
                "Feature",
                "DODA_Final_Score",
                "DODA_Rank"
            ]
        ],
        on="Feature",
        how="inner"
    )
)


# -------------------------------------------------------------------------
# Add clinical weights
# -------------------------------------------------------------------------

rank_shift_df["Clinical_Weight"] = (
    rank_shift_df["Feature"]
    .map(selector.clinical_weights_)
)


# -------------------------------------------------------------------------
# Calculate rank movement
# -------------------------------------------------------------------------
# Positive value  = moved upward
# Negative value  = moved downward
# Zero             = no rank movement

rank_shift_df["Rank_Change"] = (
    rank_shift_df["ANOVA_Rank"]
    - rank_shift_df["DODA_Rank"]
)


# -------------------------------------------------------------------------
# Sort by DODA final rank
# -------------------------------------------------------------------------

rank_shift_df = (
    rank_shift_df
    .sort_values(
        by="DODA_Rank"
    )
    .reset_index(drop=True)
)


# -------------------------------------------------------------------------
# Display
# -------------------------------------------------------------------------

print("=" * 80)
print(f"ANOVA vs DODA RANK-SHIFT ANALYSIS — TOP-{k}")
print("=" * 80)

display(rank_shift_df)

ANOVA vs DODA RANK-SHIFT ANALYSIS — TOP-10


,Feature,ANOVA_Normalized_Score,ANOVA_Rank,DODA_Final_Score,DODA_Rank,Clinical_Weight,Rank_Change
0,Bp,1.000000,1,0.032018,1,0.80,0
1,Bu,0.145469,4,0.031754,2,0.90,2
2,Sc,0.086553,8,0.031099,3,1.00,5
3,Su,0.165780,2,0.030835,4,0.55,-2
4,Ane,0.138924,5,0.030536,5,0.70,0
5,Pcv,0.087508,7,0.030077,6,0.70,1
6,Sg,0.065414,11,0.029958,7,0.85,4
7,Ba,0.155854,3,0.029958,7,0.40,-4
8,Rbcc,0.093830,6,0.029857,9,0.55,-3
9,Al,0.018301,15,0.029727,10,1.00,5


In [18]:
# =============================================================================
# STEP 16: INSPECT CLINICAL WEIGHTS
# =============================================================================

clinical_weight_table = pd.DataFrame({
    "Feature": list(selector.clinical_weights_.keys()),
    "Clinical_Weight": list(selector.clinical_weights_.values())
}).sort_values(
    "Clinical_Weight",
    ascending=False
).reset_index(drop=True)

display(clinical_weight_table)


,Feature,Clinical_Weight
0,Al,1.00
1,Sc,1.00
2,Bu,0.90
3,Sg,0.85
4,Htn,0.85
5,Dm,0.80
6,Bp,0.80
7,Hemo,0.75
8,Pcv,0.70
9,Ane,0.70


In [19]:
# =============================================================================
# STEP 17: DODA MODEL EVALUATION — FIXED SPLIT
# =============================================================================

fixed_doda_rows = []

for top_k in TOP_K_VALUES:

    Xtr_selected = rank_doda_results[top_k]["X_train"]
    Xte_selected = rank_doda_results[top_k]["X_test"]
    selected_features = rank_doda_results[top_k]["features"]

    for model_name, model_template in models.items():

        model = model_template.__class__(
            **model_template.get_params()
        )

        Xtr = Xtr_selected
        Xte = Xte_selected

        if model_name == "Logistic Regression":
            scaler = StandardScaler()
            Xtr = scaler.fit_transform(Xtr)
            Xte = scaler.transform(Xte)

        model.fit(Xtr, y_train)

        y_pred = model.predict(Xte)
        y_prob = model.predict_proba(Xte)[:, 1]

        fixed_doda_rows.append({
            "Method": "DODA",
            "Top_K": top_k,
            "Model": model_name,
            "Selected_Features": ", ".join(selected_features),
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred, zero_division=0),
            "Recall": recall_score(y_test, y_pred, zero_division=0),
            "F1": f1_score(y_test, y_pred, zero_division=0),
            "ROC_AUC": roc_auc_score(y_test, y_prob)
        })

fixed_doda_results_df = pd.DataFrame(fixed_doda_rows)
display(fixed_doda_results_df.round(4))


,Method,Top_K,Model,Selected_Features,Accuracy,Precision,Recall,F1,ROC_AUC
0,DODA,5,Logistic Regression,"Bp, Bu, Sc, Su, Ane",0.5303,0.5378,0.6275,0.5792,0.5518
1,DODA,5,Random Forest,"Bp, Bu, Sc, Su, Ane",0.4798,0.4949,0.4804,0.4876,0.4952
2,DODA,5,XGBoost,"Bp, Bu, Sc, Su, Ane",0.4848,0.5000,0.5294,0.5143,0.5043
3,DODA,10,Logistic Regression,"Bp, Bu, Sc, Su, Ane, Pcv, Sg, Ba, Rbcc, Al",0.5354,0.5431,0.6176,0.5780,0.5774
4,DODA,10,Random Forest,"Bp, Bu, Sc, Su, Ane, Pcv, Sg, Ba, Rbcc, Al",0.5253,0.5417,0.5098,0.5253,0.5347
5,DODA,10,XGBoost,"Bp, Bu, Sc, Su, Ane, Pcv, Sg, Ba, Rbcc, Al",0.5202,0.5385,0.4804,0.5078,0.5088
6,DODA,15,Logistic Regression,"Bp, Bu, Sc, Su, Ane, Pcv, Sg, Ba, Rbcc, Al, Dm...",0.5657,0.5727,0.6176,0.5943,0.5790
7,DODA,15,Random Forest,"Bp, Bu, Sc, Su, Ane, Pcv, Sg, Ba, Rbcc, Al, Dm...",0.5152,0.5312,0.5000,0.5152,0.5408
8,DODA,15,XGBoost,"Bp, Bu, Sc, Su, Ane, Pcv, Sg, Ba, Rbcc, Al, Dm...",0.5101,0.5258,0.5000,0.5126,0.5424
9,DODA,20,Logistic Regression,"Bp, Bu, Sc, Su, Ane, Pcv, Sg, Ba, Rbcc, Al, Dm...",0.5606,0.5701,0.5980,0.5837,0.5777


In [20]:
# =============================================================================
# STEP 18: FIXED-SPLIT ANOVA vs DODA COMPARISON
# =============================================================================

fixed_comparison_df = pd.concat(
    [fixed_anova_results_df, fixed_doda_results_df],
    ignore_index=True
)

display(fixed_comparison_df.round(4))


,Method,Top_K,Model,Selected_Features,Accuracy,Precision,Recall,F1,ROC_AUC
0,ANOVA,5,Logistic Regression,"Bp, Su, Ba, Bu, Ane",0.5303,0.5391,0.6078,0.5714,0.5587
1,ANOVA,5,Random Forest,"Bp, Su, Ba, Bu, Ane",0.5303,0.5474,0.5098,0.5279,0.5600
2,ANOVA,5,XGBoost,"Bp, Su, Ba, Bu, Ane",0.5000,0.5143,0.5294,0.5217,0.5036
3,ANOVA,10,Logistic Regression,"Bp, Su, Ba, Bu, Ane, Rbcc, Pcv, Sc, Appet, Bgr",0.5606,0.5630,0.6569,0.6063,0.5895
4,ANOVA,10,Random Forest,"Bp, Su, Ba, Bu, Ane, Rbcc, Pcv, Sc, Appet, Bgr",0.5051,0.5208,0.4902,0.5051,0.5129
5,ANOVA,10,XGBoost,"Bp, Su, Ba, Bu, Ane, Rbcc, Pcv, Sc, Appet, Bgr",0.4848,0.5000,0.4804,0.4900,0.4931
6,ANOVA,15,Logistic Regression,"Bp, Su, Ba, Bu, Ane, Rbcc, Pcv, Sc, Appet, Bgr...",0.5556,0.5660,0.5882,0.5769,0.5684
7,ANOVA,15,Random Forest,"Bp, Su, Ba, Bu, Ane, Rbcc, Pcv, Sc, Appet, Bgr...",0.5556,0.5686,0.5686,0.5686,0.5510
8,ANOVA,15,XGBoost,"Bp, Su, Ba, Bu, Ane, Rbcc, Pcv, Sc, Appet, Bgr...",0.5051,0.5204,0.5000,0.5100,0.5314
9,ANOVA,20,Logistic Regression,"Bp, Su, Ba, Bu, Ane, Rbcc, Pcv, Sc, Appet, Bgr...",0.5505,0.5596,0.5980,0.5782,0.5751


In [21]:
# =============================================================================
# STEP 19: COMPARE FEATURE SETS ON THE FIXED SPLIT
# =============================================================================

fixed_feature_comparison = []

for top_k in TOP_K_VALUES:

    anova_set = set(anova_results[top_k]["features"])
    doda_set = set(rank_doda_results[top_k]["features"])

    intersection = anova_set & doda_set
    union = anova_set | doda_set

    jaccard = len(intersection) / len(union)

    fixed_feature_comparison.append({
        "Top_K": top_k,
        "ANOVA_Features": ", ".join(anova_results[top_k]["features"]),
        "DODA_Features": ", ".join(rank_doda_results[top_k]["features"]),
        "Same_Feature_Set": anova_set == doda_set,
        "Overlap_Count": len(intersection),
        "Jaccard": jaccard,
        "Features_Changed": top_k - len(intersection)
    })

fixed_feature_comparison_df = pd.DataFrame(fixed_feature_comparison)

display(fixed_feature_comparison_df)


,Top_K,ANOVA_Features,DODA_Features,Same_Feature_Set,Overlap_Count,Jaccard,Features_Changed
0,5,"Bp, Su, Ba, Bu, Ane","Bp, Bu, Sc, Su, Ane",False,4,0.666667,1
1,10,"Bp, Su, Ba, Bu, Ane, Rbcc, Pcv, Sc, Appet, Bgr","Bp, Bu, Sc, Su, Ane, Pcv, Sg, Ba, Rbcc, Al",False,8,0.666667,2
2,15,"Bp, Su, Ba, Bu, Ane, Rbcc, Pcv, Sc, Appet, Bgr...","Bp, Bu, Sc, Su, Ane, Pcv, Sg, Ba, Rbcc, Al, Dm...",False,14,0.875000,1
3,20,"Bp, Su, Ba, Bu, Ane, Rbcc, Pcv, Sc, Appet, Bgr...","Bp, Bu, Sc, Su, Ane, Pcv, Sg, Ba, Rbcc, Al, Dm...",False,18,0.818182,2


In [22]:
# =============================================================================
# STEP 20: SAVE FIXED-SPLIT DODA RESULTS
# =============================================================================

doda_dir = Path("../../../results/kidney_disease/doda")
doda_dir.mkdir(parents=True, exist_ok=True)

fixed_doda_results_df.to_csv(
    doda_dir / "anova_doda_fixed_split_results.csv",
    index=False
)

fixed_feature_comparison_df.to_csv(
    doda_dir / "anova_doda_fixed_split_feature_comparison.csv",
    index=False
)

print(f"Saved results to: {doda_dir.resolve()}")


Saved results to: /home/claude/bd_kdd_final/results/kidney_disease/doda


# 3. Robustness: 5×5 Repeated Stratified Cross-Validation

The fixed split provides a reproducible initial comparison.

For robustness, feature selection is **recomputed inside every training fold**. This prevents information from the validation fold from influencing ANOVA or DODA.

There are:

- 5 folds
- 5 repeats
- 25 train/validation runs
- 4 Top-K budgets
- 2 feature-selection methods
- 3 downstream models

This produces 600 model-evaluation rows.


In [23]:
# =============================================================================
# STEP 21: REPEATED CV CONFIGURATION
# =============================================================================

from sklearn.model_selection import RepeatedStratifiedKFold

TOP_K_VALUES = [5, 10, 15, 20]

N_SPLITS = 5
N_REPEATS = 5
RANDOM_STATE = 42

cv = RepeatedStratifiedKFold(
    n_splits=N_SPLITS,
    n_repeats=N_REPEATS,
    random_state=RANDOM_STATE
)

print(f"Top-K values: {TOP_K_VALUES}")
print(f"CV runs: {N_SPLITS * N_REPEATS}")


Top-K values: [5, 10, 15, 20]
CV runs: 25


In [24]:
# =============================================================================
# STEP 22: 5×5 REPEATED CV — ANOVA vs DODA
# =============================================================================

cv_results = []

for run_id, (train_idx, val_idx) in enumerate(
    cv.split(X, y),
    start=1
):

    X_train_fold = X.iloc[train_idx].copy()
    X_val_fold = X.iloc[val_idx].copy()

    y_train_fold = y.iloc[train_idx].copy()
    y_val_fold = y.iloc[val_idx].copy()

    print(f"Run {run_id:02d}/{N_SPLITS * N_REPEATS}")

    # -----------------------------------------------------------------
    # ANOVA scores ALL features inside this training fold
    # -----------------------------------------------------------------

    anova_all = SelectKBest(
        score_func=f_classif,
        k="all"
    )

    anova_all.fit(
        X_train_fold,
        y_train_fold
    )

    anova_fold_scores = pd.DataFrame({
        "Feature": X_train_fold.columns,
        "Score": anova_all.scores_
    }).sort_values(
        "Score",
        ascending=False
    )

    # -----------------------------------------------------------------
    # DODA also receives ALL features and performs final Top-K
    # -----------------------------------------------------------------

    # NOTE: previously this cell also fit a `doda_selector` once here at
    # top_k=max(TOP_K_VALUES) before the loop below -- that result was
    # never used (the loop always builds its own fold_doda per top_k),
    # so the redundant fit was removed to cut this cell's runtime.

    for top_k in TOP_K_VALUES:

        # -------------------------
        # ANOVA Top-K
        # -------------------------

        anova_features = (
            anova_fold_scores
            .head(top_k)["Feature"]
            .tolist()
        )

        # -------------------------
        # DODA Top-K
        # -------------------------

        fold_doda_operator = SklearnAdapter(
            SelectKBest(
                score_func=f_classif,
                k="all"
            )
        )

        fold_doda = DODASelector(
            operators=[fold_doda_operator],
            provider=NestedJSONProvider(str(WEIGHTS_PATH)),
            fusion=RankFusion(),
            top_k=top_k
        )

        fold_doda.fit(
            X_train_fold,
            y_train_fold
        )

        doda_features = list(
            fold_doda.get_selected_features()
        )

        # -------------------------
        # Model loop
        # -------------------------

        feature_sets = {
            "ANOVA": anova_features,
            "DODA": doda_features
        }

        for method, selected_features in feature_sets.items():

            Xtr_selected = X_train_fold[selected_features]
            Xval_selected = X_val_fold[selected_features]

            for model_name, model_template in models.items():

                model = model_template.__class__(
                    **model_template.get_params()
                )

                Xtr = Xtr_selected
                Xval = Xval_selected

                if model_name == "Logistic Regression":
                    scaler = StandardScaler()
                    Xtr = scaler.fit_transform(Xtr)
                    Xval = scaler.transform(Xval)

                model.fit(Xtr, y_train_fold)

                y_pred = model.predict(Xval)
                y_prob = model.predict_proba(Xval)[:, 1]

                cv_results.append({
                    "Run": run_id,
                    "Method": method,
                    "Top_K": top_k,
                    "Model": model_name,
                    "Accuracy": accuracy_score(y_val_fold, y_pred),
                    "Precision": precision_score(
                        y_val_fold, y_pred, zero_division=0
                    ),
                    "Recall": recall_score(
                        y_val_fold, y_pred, zero_division=0
                    ),
                    "F1": f1_score(
                        y_val_fold, y_pred, zero_division=0
                    ),
                    "ROC_AUC": roc_auc_score(
                        y_val_fold, y_prob
                    ),
                    "Selected_Features": ", ".join(selected_features)
                })

cv_results_df = pd.DataFrame(cv_results)

print("Completed.")
print(f"Rows generated: {len(cv_results_df)}")

assert len(cv_results_df) == 25 * 4 * 2 * 3

display(cv_results_df.head())


Run 01/25
Provider inside knowledge engine: <__main__.NestedJSONProvider object at 0x7f58891888f0>
DODA Pipeline Started
Running: SelectKBest
DODA feature order:
['Age', 'Bp', 'Sg', 'Al', 'Su', 'Rbc', 'Pc', 'Pcc', 'Ba', 'Bgr']
Number of features: 24
Number of scores: 24
Operator Scores:
{'SelectKBest_1': {'Age': np.float64(0.5692921565173608), 'Bp': np.float64(8.782731827841262), 'Sg': np.float64(3.670322367176968), 'Al': np.float64(0.0457132428955518), 'Su': np.float64(1.7944164384023895), 'Rbc': np.float64(0.3016609605897061), 'Pc': np.float64(0.3327267055767738), 'Pcc': np.float64(0.013120250838075088), 'Ba': np.float64(1.0578743890740132), 'Bgr': np.float64(0.6151995671004556), 'Bu': np.float64(8.565899281370292e-05), 'Sc': np.float64(0.8746376579674088), 'Sod': np.float64(0.6491999804114769), 'Pot': np.float64(0.9935838942402748), 'Hemo': np.float64(0.5374636482028966), 'Pcv': np.float64(0.11476801752744284), 'Wbcc': np.float64(1.0518558894081236), 'Rbcc': np.float64(2.18867159921

Provider inside knowledge engine: <__main__.NestedJSONProvider object at 0x7f5889171190>
DODA Pipeline Started
Running: SelectKBest
DODA feature order:
['Age', 'Bp', 'Sg', 'Al', 'Su', 'Rbc', 'Pc', 'Pcc', 'Ba', 'Bgr']
Number of features: 24
Number of scores: 24
Operator Scores:
{'SelectKBest_1': {'Age': np.float64(0.5692921565173608), 'Bp': np.float64(8.782731827841262), 'Sg': np.float64(3.670322367176968), 'Al': np.float64(0.0457132428955518), 'Su': np.float64(1.7944164384023895), 'Rbc': np.float64(0.3016609605897061), 'Pc': np.float64(0.3327267055767738), 'Pcc': np.float64(0.013120250838075088), 'Ba': np.float64(1.0578743890740132), 'Bgr': np.float64(0.6151995671004556), 'Bu': np.float64(8.565899281370292e-05), 'Sc': np.float64(0.8746376579674088), 'Sod': np.float64(0.6491999804114769), 'Pot': np.float64(0.9935838942402748), 'Hemo': np.float64(0.5374636482028966), 'Pcv': np.float64(0.11476801752744284), 'Wbcc': np.float64(1.0518558894081236), 'Rbcc': np.float64(2.1886715992195693), 'H

Provider inside knowledge engine: <__main__.NestedJSONProvider object at 0x7f5889189910>
DODA Pipeline Started
Running: SelectKBest
DODA feature order:
['Age', 'Bp', 'Sg', 'Al', 'Su', 'Rbc', 'Pc', 'Pcc', 'Ba', 'Bgr']
Number of features: 24
Number of scores: 24
Operator Scores:
{'SelectKBest_1': {'Age': np.float64(0.5692921565173608), 'Bp': np.float64(8.782731827841262), 'Sg': np.float64(3.670322367176968), 'Al': np.float64(0.0457132428955518), 'Su': np.float64(1.7944164384023895), 'Rbc': np.float64(0.3016609605897061), 'Pc': np.float64(0.3327267055767738), 'Pcc': np.float64(0.013120250838075088), 'Ba': np.float64(1.0578743890740132), 'Bgr': np.float64(0.6151995671004556), 'Bu': np.float64(8.565899281370292e-05), 'Sc': np.float64(0.8746376579674088), 'Sod': np.float64(0.6491999804114769), 'Pot': np.float64(0.9935838942402748), 'Hemo': np.float64(0.5374636482028966), 'Pcv': np.float64(0.11476801752744284), 'Wbcc': np.float64(1.0518558894081236), 'Rbcc': np.float64(2.1886715992195693), 'H

Provider inside knowledge engine: <__main__.NestedJSONProvider object at 0x7f5892dae1b0>
DODA Pipeline Started
Running: SelectKBest
DODA feature order:
['Age', 'Bp', 'Sg', 'Al', 'Su', 'Rbc', 'Pc', 'Pcc', 'Ba', 'Bgr']
Number of features: 24
Number of scores: 24
Operator Scores:
{'SelectKBest_1': {'Age': np.float64(0.5692921565173608), 'Bp': np.float64(8.782731827841262), 'Sg': np.float64(3.670322367176968), 'Al': np.float64(0.0457132428955518), 'Su': np.float64(1.7944164384023895), 'Rbc': np.float64(0.3016609605897061), 'Pc': np.float64(0.3327267055767738), 'Pcc': np.float64(0.013120250838075088), 'Ba': np.float64(1.0578743890740132), 'Bgr': np.float64(0.6151995671004556), 'Bu': np.float64(8.565899281370292e-05), 'Sc': np.float64(0.8746376579674088), 'Sod': np.float64(0.6491999804114769), 'Pot': np.float64(0.9935838942402748), 'Hemo': np.float64(0.5374636482028966), 'Pcv': np.float64(0.11476801752744284), 'Wbcc': np.float64(1.0518558894081236), 'Rbcc': np.float64(2.1886715992195693), 'H

Run 02/25
Provider inside knowledge engine: <__main__.NestedJSONProvider object at 0x7f5889188a10>
DODA Pipeline Started
Running: SelectKBest
DODA feature order:
['Age', 'Bp', 'Sg', 'Al', 'Su', 'Rbc', 'Pc', 'Pcc', 'Ba', 'Bgr']
Number of features: 24
Number of scores: 24
Operator Scores:
{'SelectKBest_1': {'Age': np.float64(0.004887964692271122), 'Bp': np.float64(8.803218382772398), 'Sg': np.float64(0.6219324765632036), 'Al': np.float64(0.715171824866349), 'Su': np.float64(0.319162875750569), 'Rbc': np.float64(0.00999357570923964), 'Pc': np.float64(0.5898716201928145), 'Pcc': np.float64(0.36906995636015666), 'Ba': np.float64(4.130454254359554), 'Bgr': np.float64(0.0003840765981331331), 'Bu': np.float64(0.6784571599823763), 'Sc': np.float64(0.6685011460240117), 'Sod': np.float64(0.4111113591263141), 'Pot': np.float64(1.7470281519321484), 'Hemo': np.float64(0.10225469245919629), 'Pcv': np.float64(3.3689465419322278), 'Wbcc': np.float64(0.5892010793111393), 'Rbcc': np.float64(2.54607054896

...

Provider inside knowledge engine: <__main__.NestedJSONProvider object at 0x7f58809c1d90>
DODA Pipeline Started
Running: SelectKBest
DODA feature order:
['Age', 'Bp', 'Sg', 'Al', 'Su', 'Rbc', 'Pc', 'Pcc', 'Ba', 'Bgr']
Number of features: 24
Number of scores: 24
Operator Scores:
{'SelectKBest_1': {'Age': np.float64(0.30476586839344877), 'Bp': np.float64(8.607825202112682), 'Sg': np.float64(3.6981271928676813), 'Al': np.float64(0.6286812414619772), 'Su': np.float64(3.268751990164252), 'Rbc': np.float64(0.018472198222493793), 'Pc': np.float64(0.034641131531844345), 'Pcc': np.float64(0.018472198222493793), 'Ba': np.float64(1.0117930948604392), 'Bgr': np.float64(0.37399213166783357), 'Bu': np.float64(3.216502972170651), 'Sc': np.float64(1.0205138391147865), 'Sod': np.float64(0.05177592565176037), 'Pot': np.float64(0.6464991838290113), 'Hemo': np.float64(0.42481807602448285), 'Pcv': np.float64(1.0149476121709906), 'Wbcc': np.float64(0.12927247346268472), 'Rbcc': np.float64(2.4332346628792925)

Provider inside knowledge engine: <__main__.NestedJSONProvider object at 0x7f5889149040>
DODA Pipeline Started
Running: SelectKBest
DODA feature order:
['Age', 'Bp', 'Sg', 'Al', 'Su', 'Rbc', 'Pc', 'Pcc', 'Ba', 'Bgr']
Number of features: 24
Number of scores: 24
Operator Scores:
{'SelectKBest_1': {'Age': np.float64(0.30476586839344877), 'Bp': np.float64(8.607825202112682), 'Sg': np.float64(3.6981271928676813), 'Al': np.float64(0.6286812414619772), 'Su': np.float64(3.268751990164252), 'Rbc': np.float64(0.018472198222493793), 'Pc': np.float64(0.034641131531844345), 'Pcc': np.float64(0.018472198222493793), 'Ba': np.float64(1.0117930948604392), 'Bgr': np.float64(0.37399213166783357), 'Bu': np.float64(3.216502972170651), 'Sc': np.float64(1.0205138391147865), 'Sod': np.float64(0.05177592565176037), 'Pot': np.float64(0.6464991838290113), 'Hemo': np.float64(0.42481807602448285), 'Pcv': np.float64(1.0149476121709906), 'Wbcc': np.float64(0.12927247346268472), 'Rbcc': np.float64(2.4332346628792925)

Provider inside knowledge engine: <__main__.NestedJSONProvider object at 0x7f58891490a0>
DODA Pipeline Started
Running: SelectKBest
DODA feature order:
['Age', 'Bp', 'Sg', 'Al', 'Su', 'Rbc', 'Pc', 'Pcc', 'Ba', 'Bgr']
Number of features: 24
Number of scores: 24
Operator Scores:
{'SelectKBest_1': {'Age': np.float64(0.30476586839344877), 'Bp': np.float64(8.607825202112682), 'Sg': np.float64(3.6981271928676813), 'Al': np.float64(0.6286812414619772), 'Su': np.float64(3.268751990164252), 'Rbc': np.float64(0.018472198222493793), 'Pc': np.float64(0.034641131531844345), 'Pcc': np.float64(0.018472198222493793), 'Ba': np.float64(1.0117930948604392), 'Bgr': np.float64(0.37399213166783357), 'Bu': np.float64(3.216502972170651), 'Sc': np.float64(1.0205138391147865), 'Sod': np.float64(0.05177592565176037), 'Pot': np.float64(0.6464991838290113), 'Hemo': np.float64(0.42481807602448285), 'Pcv': np.float64(1.0149476121709906), 'Wbcc': np.float64(0.12927247346268472), 'Rbcc': np.float64(2.4332346628792925)

Completed.
Rows generated: 600


,Run,Method,Top_K,Model,Accuracy,Precision,Recall,F1,ROC_AUC,Selected_Features
0,1,ANOVA,5,Logistic Regression,0.525253,0.530973,0.594059,0.560748,0.524497,"Bp, Sg, Rbcc, Su, Ba"
1,1,ANOVA,5,Random Forest,0.510101,0.518868,0.544554,0.531401,0.503062,"Bp, Sg, Rbcc, Su, Ba"
2,1,ANOVA,5,XGBoost,0.515152,0.524752,0.524752,0.524752,0.499898,"Bp, Sg, Rbcc, Su, Ba"
3,1,DODA,5,Logistic Regression,0.500000,0.509615,0.524752,0.517073,0.518322,"Bp, Sg, Sc, Rbcc, Su"
4,1,DODA,5,Random Forest,0.525253,0.535354,0.524752,0.530000,0.539145,"Bp, Sg, Sc, Rbcc, Su"


In [25]:
# =============================================================================
# STEP 23: CV PERFORMANCE SUMMARY
# =============================================================================

cv_summary_df = (
    cv_results_df
    .groupby(["Method", "Top_K", "Model"])
    [["Accuracy", "Precision", "Recall", "F1", "ROC_AUC"]]
    .agg(["mean", "std"])
    .reset_index()
)

cv_summary_df.columns = [
    "Method", "Top_K", "Model",
    "Accuracy_Mean", "Accuracy_STD",
    "Precision_Mean", "Precision_STD",
    "Recall_Mean", "Recall_STD",
    "F1_Mean", "F1_STD",
    "ROC_AUC_Mean", "ROC_AUC_STD"
]

display(cv_summary_df.round(4))


,Method,Top_K,Model,Accuracy_Mean,Accuracy_STD,Precision_Mean,Precision_STD,Recall_Mean,Recall_STD,F1_Mean,F1_STD,ROC_AUC_Mean,ROC_AUC_STD
0,ANOVA,5,Logistic Regression,0.5170,0.0341,0.5264,0.0294,0.5823,0.0507,0.5525,0.0360,0.5191,0.0356
1,ANOVA,5,Random Forest,0.5093,0.0293,0.5216,0.0296,0.5289,0.0503,0.5244,0.0353,0.5049,0.0368
2,ANOVA,5,XGBoost,0.5042,0.0372,0.5157,0.0366,0.5400,0.0551,0.5270,0.0433,0.4977,0.0422
3,ANOVA,10,Logistic Regression,0.5180,0.0380,0.5284,0.0345,0.5657,0.0532,0.5458,0.0399,0.5216,0.0389
4,ANOVA,10,Random Forest,0.4976,0.0307,0.5108,0.0316,0.5306,0.0409,0.5198,0.0300,0.4929,0.0460
5,ANOVA,10,XGBoost,0.5049,0.0329,0.5165,0.0317,0.5409,0.0570,0.5276,0.0403,0.4962,0.0396
6,ANOVA,15,Logistic Regression,0.5204,0.0282,0.5307,0.0262,0.5613,0.0511,0.5449,0.0345,0.5221,0.0357
7,ANOVA,15,Random Forest,0.5061,0.0365,0.5185,0.0341,0.5487,0.0458,0.5324,0.0344,0.5079,0.0402
8,ANOVA,15,XGBoost,0.5087,0.0397,0.5203,0.0367,0.5444,0.0546,0.5315,0.0421,0.5119,0.0472
9,ANOVA,20,Logistic Regression,0.5162,0.0328,0.5269,0.0306,0.5578,0.0493,0.5414,0.0363,0.5207,0.0339


In [26]:
# =============================================================================
# STEP 24: SAVE REPEATED-CV RESULTS
# =============================================================================

cv_dir = Path("../../../results/kidney_disease/cv")
cv_dir.mkdir(parents=True, exist_ok=True)

cv_results_df.to_csv(
    cv_dir / "anova_vs_doda_5x5_repeated_cv_results.csv",
    index=False
)

cv_summary_df.to_csv(
    cv_dir / "anova_vs_doda_5x5_repeated_cv_summary.csv",
    index=False
)

print(f"Saved to: {cv_dir.resolve()}")


Saved to: /home/claude/bd_kdd_final/results/kidney_disease/cv


# 4. Feature-set Stability and Rank/Set Change

Two different concepts are reported:

- **Feature-set change:** whether the Top-K membership changes.
- **Rank change:** whether feature ordering changes even when membership is retained.

Jaccard similarity evaluates set overlap, so it does not measure ordering. The fixed-split DODA rank comparison is used to inspect ordering changes.


In [27]:
# =============================================================================
# STEP 25: FEATURE-SET COMPARISON ACROSS THE 25 CV RUNS
# =============================================================================

feature_comparison_rows = []

for run_id in range(1, 26):

    for top_k in TOP_K_VALUES:

        anova_row = cv_results_df[
            (cv_results_df["Run"] == run_id) &
            (cv_results_df["Method"] == "ANOVA") &
            (cv_results_df["Top_K"] == top_k)
        ].iloc[0]

        doda_row = cv_results_df[
            (cv_results_df["Run"] == run_id) &
            (cv_results_df["Method"] == "DODA") &
            (cv_results_df["Top_K"] == top_k)
        ].iloc[0]

        anova_set = set(
            anova_row["Selected_Features"].split(", ")
        )
        doda_set = set(
            doda_row["Selected_Features"].split(", ")
        )

        overlap = len(anova_set & doda_set)
        union = len(anova_set | doda_set)

        feature_comparison_rows.append({
            "Run": run_id,
            "Top_K": top_k,
            "Same_Feature_Set": anova_set == doda_set,
            "Overlap_Count": overlap,
            "Jaccard": overlap / union,
            "Features_Changed": top_k - overlap
        })

feature_comparison_df = pd.DataFrame(
    feature_comparison_rows
)

display(
    feature_comparison_df
    .groupby("Top_K")
    [["Same_Feature_Set", "Overlap_Count", "Jaccard", "Features_Changed"]]
    .agg(["mean", "std"])
    .round(4)
)


Same_Feature_Set         Overlap_Count         Jaccard          \
                  mean     std          mean     std    mean     std   
Top_K                                                                  
5                 0.04  0.2000          3.60  0.6455  0.5776  0.1577   
10                0.00  0.0000          8.52  0.5859  0.7464  0.0863   
15                0.16  0.3742         13.40  0.9574  0.8131  0.1069   
20                0.00  0.0000         18.32  0.8021  0.8474  0.0669   

      Features_Changed          
                  mean     std  
Top_K                           
5                 1.40  0.6455  
10                1.48  0.5859  
15                1.60  0.9574  
20                1.68  0.8021

In [28]:
# =============================================================================
# STEP 26: FEATURE-SELECTION STABILITY
# =============================================================================

from itertools import combinations

stability_rows = []

for method in ["ANOVA", "DODA"]:

    for top_k in TOP_K_VALUES:

        subset = cv_results_df[
            (cv_results_df["Method"] == method) &
            (cv_results_df["Top_K"] == top_k)
        ]

        # One feature set per run; model does not affect selection.
        run_sets = {}

        for run_id in range(1, 26):
            row = subset[
                subset["Run"] == run_id
            ].iloc[0]

            run_sets[run_id] = set(
                row["Selected_Features"].split(", ")
            )

        pairwise_jaccards = []

        for a, b in combinations(range(1, 26), 2):

            set_a = run_sets[a]
            set_b = run_sets[b]

            pairwise_jaccards.append(
                len(set_a & set_b) / len(set_a | set_b)
            )

        stability_rows.append({
            "Method": method,
            "Top_K": top_k,
            "Mean_Jaccard": np.mean(pairwise_jaccards),
            "SD_Jaccard": np.std(pairwise_jaccards, ddof=1),
            "Min_Jaccard": np.min(pairwise_jaccards),
            "Max_Jaccard": np.max(pairwise_jaccards)
        })

stability_df = pd.DataFrame(stability_rows)

display(stability_df.round(4))


,Method,Top_K,Mean_Jaccard,SD_Jaccard,Min_Jaccard,Max_Jaccard
0,ANOVA,5,0.4218,0.1710,0.1111,1.0000
1,ANOVA,10,0.5030,0.1192,0.2500,0.8182
2,ANOVA,15,0.6260,0.1043,0.3636,1.0000
3,ANOVA,20,0.7676,0.0618,0.6667,0.9048
4,DODA,5,0.3910,0.1674,0.1111,1.0000
5,DODA,10,0.5177,0.1259,0.2500,1.0000
6,DODA,15,0.6534,0.1080,0.4286,0.8750
7,DODA,20,0.8245,0.0676,0.6667,1.0000


In [29]:
# =============================================================================
# STEP 27: PAIRED STATISTICAL COMPARISON OF PREDICTIVE PERFORMANCE
# =============================================================================

from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

metrics = ["Accuracy", "Precision", "Recall", "F1", "ROC_AUC"]

comparison_rows = []

for top_k in TOP_K_VALUES:

    for model_name in models.keys():

        anova = cv_results_df[
            (cv_results_df["Method"] == "ANOVA") &
            (cv_results_df["Top_K"] == top_k) &
            (cv_results_df["Model"] == model_name)
        ].sort_values("Run")

        doda = cv_results_df[
            (cv_results_df["Method"] == "DODA") &
            (cv_results_df["Top_K"] == top_k) &
            (cv_results_df["Model"] == model_name)
        ].sort_values("Run")

        assert len(anova) == 25
        assert len(doda) == 25
        assert (anova["Run"].values == doda["Run"].values).all()

        for metric in metrics:

            anova_values = anova[metric].to_numpy()
            doda_values = doda[metric].to_numpy()

            differences = doda_values - anova_values

            mean_difference = np.mean(differences)
            sd_difference = np.std(differences, ddof=1)

            if np.allclose(differences, 0):
                statistic = 0.0
                p_value = 1.0
            else:
                statistic, p_value = wilcoxon(
                    doda_values,
                    anova_values,
                    alternative="two-sided"
                )

            paired_d = (
                mean_difference / sd_difference
                if sd_difference > 0 else 0.0
            )

            comparison_rows.append({
                "Top_K": top_k,
                "Model": model_name,
                "Metric": metric,
                "ANOVA_Mean": np.mean(anova_values),
                "DODA_Mean": np.mean(doda_values),
                "Mean_Difference_DODA_minus_ANOVA": mean_difference,
                "Paired_SD": sd_difference,
                "Paired_Cohens_d": paired_d,
                "Wilcoxon_Statistic": statistic,
                "P_Value": p_value
            })

performance_comparison_df = pd.DataFrame(comparison_rows)

reject, adjusted_p, _, _ = multipletests(
    performance_comparison_df["P_Value"],
    alpha=0.05,
    method="holm"
)

performance_comparison_df["Holm_Adjusted_P"] = adjusted_p
performance_comparison_df["Significant"] = reject

display(performance_comparison_df.round(4))


,Top_K,Model,Metric,ANOVA_Mean,DODA_Mean,Mean_Difference_DODA_minus_ANOVA,Paired_SD,Paired_Cohens_d,Wilcoxon_Statistic,P_Value,Holm_Adjusted_P,Significant
0,5,Logistic Regression,Accuracy,0.5170,0.5150,-0.0020,0.0269,-0.0754,112.0,0.6377,1.0,False
1,5,Logistic Regression,Precision,0.5264,0.5242,-0.0022,0.0236,-0.0938,130.0,0.5677,1.0,False
2,5,Logistic Regression,Recall,0.5823,0.5814,-0.0009,0.0543,-0.0158,113.0,0.6611,1.0,False
3,5,Logistic Regression,F1,0.5525,0.5507,-0.0018,0.0350,-0.0513,149.0,0.9772,1.0,False
4,5,Logistic Regression,ROC_AUC,0.5191,0.5161,-0.0030,0.0148,-0.2034,129.0,0.5485,1.0,False
5,5,Random Forest,Accuracy,0.5093,0.5026,-0.0067,0.0354,-0.1886,131.0,0.5869,1.0,False
6,5,Random Forest,Precision,0.5216,0.5153,-0.0064,0.0341,-0.1864,143.0,0.6150,1.0,False
7,5,Random Forest,Recall,0.5289,0.5242,-0.0047,0.0528,-0.0896,114.5,0.6967,1.0,False
8,5,Random Forest,F1,0.5244,0.5190,-0.0055,0.0382,-0.1429,148.0,0.7112,1.0,False
9,5,Random Forest,ROC_AUC,0.5049,0.4977,-0.0072,0.0331,-0.2189,133.0,0.4418,1.0,False


In [30]:
# =============================================================================
# STEP 28: SAVE STATISTICAL RESULTS
# =============================================================================

stats_dir = Path("../../../results/kidney_disease/statistics")
stats_dir.mkdir(parents=True, exist_ok=True)

feature_comparison_df.to_csv(
    stats_dir / "feature_set_comparison_5x5.csv",
    index=False
)

stability_df.to_csv(
    stats_dir / "feature_selection_stability_5x5.csv",
    index=False
)

performance_comparison_df.to_csv(
    stats_dir / "anova_vs_doda_predictive_statistics_5x5.csv",
    index=False
)

print(f"Saved statistical outputs to: {stats_dir.resolve()}")


Saved statistical outputs to: /home/claude/bd_kdd_final/results/kidney_disease/statistics


In [31]:
# =============================================================================
# STEP 29: FINAL SANITY CHECKS
# =============================================================================

print("=" * 70)
print("FINAL EXPERIMENT SANITY CHECK")
print("=" * 70)

print(f"Total predictors: {X.shape[1]}")
print(f"Top-K values: {TOP_K_VALUES}")
print(f"Missing values: {int(X.isna().sum().sum())}")
print(f"CV result rows: {len(cv_results_df)}")
print(f"Expected CV rows: {25 * len(TOP_K_VALUES) * 2 * len(models)}")

assert X.shape[1] == 24
assert X.isna().sum().sum() == 0
assert TOP_K_VALUES == [5, 10, 15, 20]
assert len(cv_results_df) == 25 * len(TOP_K_VALUES) * 2 * len(models)

for top_k in TOP_K_VALUES:
    assert len(anova_results[top_k]["features"]) == top_k
    assert len(rank_doda_results[top_k]["features"]) == top_k

print("All sanity checks passed.")


FINAL EXPERIMENT SANITY CHECK
Total predictors: 24
Top-K values: [5, 10, 15, 20]
Missing values: 0
CV result rows: 600
Expected CV rows: 600
All sanity checks passed.


# Methodological corrections made in this final version

### 1. Correct predictor count
BD-KDD has **25 processed columns**, but one is the binary target `Class`. Therefore the feature matrix contains **24 predictors**.

### 2. Top-K values
The experiment uses:

**Top-5, Top-10, Top-15, Top-20**

This provides approximately 20%, 40%, 60%, and 80% feature budgets over the 24 predictors. Top-24 is deliberately not used as a selection condition because retaining every feature cannot demonstrate feature-selection behavior.

### 3. Removed imputation
BD-KDD has zero missing values, so no imputation is performed.

### 4. Correct DODA architecture
ANOVA is configured with `k="all"` inside DODA. DODA receives all 24 statistical scores and performs the final Top-K selection.

### 5. No data leakage in CV
ANOVA and DODA are refit inside every training fold. Validation-fold data is never used to calculate feature-selection scores.

### 6. Model-specific scaling
Scaling is applied only to Logistic Regression and is fitted on the training data/fold. Random Forest and XGBoost use the selected features without scaling.

### 7. Consistent result paths
All outputs are stored under:

```text
results/
└── kidney_disease/
    ├── baseline/
    ├── doda/
    ├── cv/
    └── statistics/
```

### 8. Fixed split and repeated CV are separated
The fixed 80/20 experiment is an initial reproducible comparison. The 5×5 repeated stratified CV is the robustness analysis.

### 9. Feature-set change and rank change are separated
Jaccard measures feature-set overlap. DODA's rank-comparison output is retained separately because a feature can change rank without entering or leaving Top-K.

### 10. Clinical weights remain external
The clinical weights are loaded from the pre-specified JSON and are not derived from ANOVA scores, target correlations, model performance, or test-fold information.


## Interpretation rule

Do not assume that DODA must improve predictive performance.

The experiment should answer three separate questions:

1. **Does DODA change feature prioritization?**
2. **How stable are ANOVA and DODA selections across repeated folds?**
3. **What happens to predictive performance when the feature priorities are changed?**

If DODA changes the selected features while predictive performance remains similar, that is a different finding from DODA improving predictive performance.

All conclusions should be based on the generated results rather than predetermined expectations.
